# 00 — Data Preparation

Pipeline preprocessing video → wajah ter-ekstraksi → `tf.data.Dataset`.

**Spesifikasi (sesuai request client):**
- Library: MTCNN + OpenCV untuk deteksi & crop wajah
- Ambil **10 frame per video** secara merata via `np.linspace`
- Output ukuran wajah **112×112** (ArcFace native input)
- Fallback: kalau MTCNN gagal deteksi → pakai wajah valid sebelumnya; kalau belum pernah ada → gambar putih
- Simpan hasil ke folder subset masing-masing, lalu serialize ke `tf.data.Dataset`

Struktur dataset asli:
```
CVPR 2017 dataset/
├── train 600/  (600 video .mp4)
├── val 200/    (200 video .mp4)
└── test 200/   (200 video .mp4)
```

In [1]:
import os
import warnings
from pathlib import Path

import numpy as np
import tensorflow as tf

from utils import (
    extract_face_from_video,
    build_dataset_from_faces,
    load_annotations,
    check_number_of_images,
    IMG_SIZE,
    NUM_FRAMES,
)

warnings.filterwarnings('ignore')
print('TensorFlow:', tf.__version__)

TensorFlow: 2.13.0


## 1. Konfigurasi path

Sesuaikan dengan struktur folder lu kalau berbeda.

In [2]:
# Folder asli dataset video
VIDEO_DIRS = {
    'train': 'dataset/train',
    'val':   'dataset/val',
    'test':  'dataset/test',
}

# Output: face crops per subset
FACE_DIRS = {
    'train': 'data/face_1k/train_600',
    'val':   'data/face_1k/val_200',
    'test':  'data/face_1k/test_200',
}

# Output: serialized tf.data.Dataset
DS_DIRS = {
    'train': 'data/face_1k/ds/train_ds',
    'val':   'data/face_1k/ds/val_ds',
    'test':  'data/face_1k/ds/test_ds',
}

for d in list(FACE_DIRS.values()) + ['data/face_1k/ds']:
    os.makedirs(d, exist_ok=True)

print('Image size :', IMG_SIZE)
print('Num frames :', NUM_FRAMES)

Image size : (112, 112)
Num frames : 10


## 2. Load anotasi OCEAN

File `.pkl` dari ChaLearn First Impressions V2 — sudah ada di folder `annotations/`.

In [3]:
ann_train, ann_val, ann_test = load_annotations('annotations')
annotations = {'train': ann_train, 'val': ann_val, 'test': ann_test}

# Sanity check
sample_key = list(ann_train['openness'].keys())[0]
print(f'Contoh video: {sample_key}')
print(f'  O={ann_train["openness"][sample_key]:.4f}')
print(f'  C={ann_train["conscientiousness"][sample_key]:.4f}')
print(f'  E={ann_train["extraversion"][sample_key]:.4f}')
print(f'  A={ann_train["agreeableness"][sample_key]:.4f}')
print(f'  N={ann_train["neuroticism"][sample_key]:.4f}')

Contoh video: J4GQm9j0JZ0.003.mp4
  O=0.4889
  C=0.6019
  E=0.5234
  A=0.6264
  N=0.5521


## 3. Ekstraksi wajah dari video (MTCNN)

**Heavy step.** Sekitar 4–5 detik per video di CPU; lebih cepat di GPU.

Jalankan per-subset. Kalau kerasa lama, bisa ambil sampling sub-set dulu (mis. 100 video) untuk testing.

In [4]:
for subset in ['train', 'val', 'test']:
    print(f'\n{"="*60}\nEkstraksi wajah subset: {subset.upper()}\n{"="*60}')
    extract_face_from_video(
        video_dir=VIDEO_DIRS[subset],
        save_dir=FACE_DIRS[subset],
        num_images=NUM_FRAMES,
        image_size=IMG_SIZE,
    )

    # Verifikasi: pastikan setiap folder punya 10 gambar
    bad = check_number_of_images(FACE_DIRS[subset], NUM_FRAMES)
    if bad:
        print(f'[WARN] {len(bad)} folder kurang dari {NUM_FRAMES} gambar:')
        for name, count in bad[:5]:
            print(f'   - {name}: {count}')
    else:
        print(f'[OK] Semua folder punya tepat {NUM_FRAMES} gambar.')


Ekstraksi wajah subset: TRAIN


Ekstraksi -> train_600:   0%|                                            | 0/600

1/1 [==============================] - 0s 493ms/step
[OK] Semua folder punya tepat 10 gambar.

Ekstraksi wajah subset: VAL


Ekstraksi -> val_200:   0%|                                              | 0/200

1/1 [==============================] - 1s 1s/step


: 

## 4. Build & save tf.data.Dataset

Format tensor:
- Input: `(num_frames=10, 112, 112, 3)` float32, range [0, 255]
- Label: `(5,)` float32, range [0, 1] (OCEAN scores)

Disimpan via `tf.data.Dataset.save()` untuk dipakai di notebook training.

In [ ]:
BATCH_SIZE = 8

for subset in ['train', 'val', 'test']:
    print(f'\n{"="*60}\nBuild dataset: {subset.upper()}\n{"="*60}')
    ds = build_dataset_from_faces(
        face_dir=FACE_DIRS[subset],
        annotation=annotations[subset],
        num_images=NUM_FRAMES,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
    )

    tf.data.Dataset.save(ds, DS_DIRS[subset])
    print(f'[OK] Disimpan ke: {DS_DIRS[subset]}')

    # Verifikasi shape
    for x, y in ds.take(1):
        print(f'  X batch shape: {x.shape}')
        print(f'  y batch shape: {y.shape}')
        print(f'  contoh label : {y[0].numpy().round(4)}')

print('\n✓ Semua dataset selesai di-generate.')

## 5. Sanity check — visualisasi 10 frame wajah

Optional. Cuma buat mastiin output MTCNN beneran wajah.

In [ ]:
import matplotlib.pyplot as plt
import cv2

sample_folder = sorted(os.listdir(FACE_DIRS['train']))[0]
sample_path   = os.path.join(FACE_DIRS['train'], sample_folder)
files = sorted(os.listdir(sample_path))

fig, axes = plt.subplots(1, NUM_FRAMES, figsize=(20, 3))
for i, f in enumerate(files[:NUM_FRAMES]):
    img = cv2.imread(os.path.join(sample_path, f))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(f'frame {i}')
    axes[i].axis('off')
plt.suptitle(f'Sample: {sample_folder}')
plt.tight_layout()
plt.show()